# Food Image Classification with EfficientNetB3
### Transfer Learning on the `bjoernjostein/food-classification` Dataset (61 Classes)

## Prerequisites

Before running this notebook, ensure you have Kaggle API credentials configured:
1. Go to https://www.kaggle.com/settings/api
2. Click **Create New Token** — this downloads `kaggle.json`
3. Place `kaggle.json` at `~/.kaggle/kaggle.json` (Linux/Mac) or `C:\\Users\\<you>\\.kaggle\\kaggle.json` (Windows)
4. On Linux/Mac: `chmod 600 ~/.kaggle/kaggle.json`

The notebook uses `kagglehub` which reads these credentials automatically.

In [ ]:
!pip install --upgrade kagglehub kagglesdk --quiet --user
!pip install timm torch torchvision numpy pandas matplotlib seaborn scikit-learn Pillow tqdm --quiet --user

## Section 1 — Setup & Imports

In [ ]:
import os
import random
import warnings
import pathlib

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, cohen_kappa_score
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision.transforms as T
import timm

warnings.filterwarnings('ignore')

In [ ]:
SEED         = 42
DATASET_SLUG = "bjoernjostein/food-classification"
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
EPOCHS       = 5 if not torch.cuda.is_available() else 20
LR           = 1e-4
FINE_TUNE_LR = 1e-5
NUM_WORKERS  = 0   # 0 is safest on Windows; increase if on Linux
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Config: EPOCHS={EPOCHS}, BATCH_SIZE={BATCH_SIZE}, DEVICE={DEVICE}")
if not torch.cuda.is_available():
    print("⚠️  No GPU detected. Training will be slow. Consider using Google Colab or reducing EPOCHS.")
else:
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")

In [ ]:
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
os.makedirs("figures", exist_ok=True)
print("Seed set. figures/ directory ready.")

## Section 2 — Data Download with kagglehub

In [ ]:
raw_path = kagglehub.dataset_download(DATASET_SLUG)
DATA_ROOT = pathlib.Path(raw_path)
print(f"Dataset downloaded to: {DATA_ROOT}")

In [ ]:
for root, dirs, files in os.walk(DATA_ROOT):
    level = str(root).replace(str(DATA_ROOT), '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{pathlib.Path(root).name}/')
    if level < 2:
        for f in files[:5]:
            print(f'{indent}  {f}')

In [ ]:
_IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def find_split_dir(root: pathlib.Path, candidates: list) -> pathlib.Path | None:
    for name in candidates:
        p = root / name
        if p.is_dir():
            return p
    return None

TRAIN_DIR = find_split_dir(DATA_ROOT, ["train", "Train", "training"])
VAL_DIR   = find_split_dir(DATA_ROOT, ["valid", "validation", "val", "Valid"])
TEST_DIR  = find_split_dir(DATA_ROOT, ["test", "Test"])

assert TRAIN_DIR is not None, "Could not find a train/ directory in the dataset"
print(f"Train dir : {TRAIN_DIR}")
print(f"Val dir   : {VAL_DIR}")
print(f"Test dir  : {TEST_DIR}")

CLASS_NAMES = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
print(f"
Found {NUM_CLASSES} classes: {CLASS_NAMES[:5]} ...")

In [ ]:
def count_images(split_dir: pathlib.Path) -> dict:
    """Return {class_name: count} for a split directory."""
    counts = {}
    for cls_dir in sorted(split_dir.iterdir()):
        if cls_dir.is_dir():
            counts[cls_dir.name] = sum(
                1 for f in cls_dir.iterdir()
                if f.suffix.lower() in _IMAGE_EXTS
            )
    return counts

train_counts = count_images(TRAIN_DIR)
val_counts   = count_images(VAL_DIR) if VAL_DIR else {}
test_counts  = count_images(TEST_DIR) if TEST_DIR else {}

print(f"Train images : {sum(train_counts.values())}")
print(f"Val images   : {sum(val_counts.values())}")
print(f"Test images  : {sum(test_counts.values())}")

In [ ]:
# Build full training file list
_all_files, _all_labels = [], []
for cls_name in CLASS_NAMES:
    cls_dir = TRAIN_DIR / cls_name
    for f in cls_dir.iterdir():
        if f.suffix.lower() in _IMAGE_EXTS:
            _all_files.append(str(f))
            _all_labels.append(CLASS_TO_IDX[cls_name])

# Carve test set from training data if no test dir exists
if TEST_DIR is None:
    print("No test/ directory found — carving 15% of training data as test set (stratified).")
    train_files, test_files, train_lbls, test_lbls = train_test_split(
        _all_files, _all_labels, test_size=0.15, random_state=SEED, stratify=_all_labels
    )
else:
    train_files, train_lbls = _all_files, _all_labels
    test_files, test_lbls = [], []
    for cls_name in CLASS_NAMES:
        test_cls = TEST_DIR / cls_name
        if test_cls.is_dir():
            for f in test_cls.iterdir():
                if f.suffix.lower() in _IMAGE_EXTS:
                    test_files.append(str(f))
                    test_lbls.append(CLASS_TO_IDX[cls_name])

# Carve val set from training data if no val dir exists
if VAL_DIR is None:
    print("No val/ directory found — carving 15% of training data as validation set (stratified).")
    train_files, val_files, train_lbls, val_lbls = train_test_split(
        train_files, train_lbls, test_size=0.15, random_state=SEED, stratify=train_lbls
    )
else:
    val_files, val_lbls = [], []
    for cls_name in CLASS_NAMES:
        val_cls = VAL_DIR / cls_name
        if val_cls.is_dir():
            for f in val_cls.iterdir():
                if f.suffix.lower() in _IMAGE_EXTS:
                    val_files.append(str(f))
                    val_lbls.append(CLASS_TO_IDX[cls_name])

print(f"
Final split sizes:")
print(f"  Train : {len(train_files)}")
print(f"  Val   : {len(val_files)}")
print(f"  Test  : {len(test_files)}")

## Section 3 — Exploratory Data Analysis (EDA)
### 3.1 Class Distribution

In [ ]:
import pandas as pd

train_series = pd.Series(train_counts).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 18))
ax.barh(train_series.index, train_series.values, color='steelblue')
ax.set_xlabel("Number of Images")
ax.set_title("Training Images per Class (sorted)")
ax.set_ylabel("Class")
plt.tight_layout()
plt.savefig("figures/class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

stats = train_series.describe()
print(f"Min: {int(stats['min'])} | Max: {int(stats['max'])} | "
      f"Mean: {stats['mean']:.1f} | Median: {stats['50%']:.1f}")
imbalance_ratio = int(stats['max']) / max(int(stats['min']), 1)
print(f"Imbalance ratio (max/min): {imbalance_ratio:.1f}x")
if imbalance_ratio > 3:
    print("⚠️  Class imbalance detected — consider class-weighted loss or oversampling.")

### 3.2 Sample Image Grid

In [ ]:
set_seed()
_sample_indices = random.sample(range(len(train_files)), min(25, len(train_files)))
_sample_files  = [train_files[i] for i in _sample_indices]
_sample_labels = [CLASS_NAMES[train_lbls[i]] for i in _sample_indices]

fig, axes = plt.subplots(5, 5, figsize=(15, 15))
for ax, fpath, label in zip(axes.flat, _sample_files, _sample_labels):
    try:
        img = Image.open(fpath).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"{label}
{img.size[0]}×{img.size[1]}", fontsize=7)
    except Exception:
        ax.set_title(f"{label}
(error)", fontsize=7, color='red')
    ax.axis('off')
fig.suptitle("Sample Training Images (5×5 Grid)", fontsize=14, y=1.01)
fig.tight_layout()
plt.savefig("figures/sample_grid.png", dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Image Dimension Analysis

In [ ]:
set_seed()
_sample_500 = random.sample(range(len(train_files)), min(500, len(train_files)))
widths, heights = [], []
for i in tqdm(_sample_500, desc="Reading image dimensions"):
    try:
        w, h = Image.open(train_files[i]).size
        widths.append(w)
        heights.append(h)
    except Exception:
        pass

ratios = [w / h for w, h in zip(widths, heights)]
aspect_buckets = [
    "square" if 0.9 <= r <= 1.1 else "portrait" if r < 0.9 else "landscape"
    for r in ratios
]
colour_map = {"square": "steelblue", "portrait": "tomato", "landscape": "seagreen"}
colours = [colour_map[b] for b in aspect_buckets]

fig, ax = plt.subplots(figsize=(8, 6))
for bucket, col in colour_map.items():
    mask = [b == bucket for b in aspect_buckets]
    ax.scatter(
        [w for w, m in zip(widths, mask) if m],
        [h for h, m in zip(heights, mask) if m],
        c=col, label=bucket, alpha=0.5, s=20
    )
ax.set_xlabel("Width (px)")
ax.set_ylabel("Height (px)")
ax.set_title("Image Dimensions (sample of 500)")
ax.legend()
plt.tight_layout()
plt.savefig("figures/dimension_scatter.png", dpi=150, bbox_inches='tight')
plt.show()

from collections import Counter
res_counts = Counter(zip(widths, heights))
most_common_res, most_common_count = res_counts.most_common(1)[0]
print(f"Most common resolution: {most_common_res[0]}×{most_common_res[1]} "
      f"({most_common_count / len(widths) * 100:.1f}% of sample)")

### 3.4 Pixel Intensity Distribution

In [ ]:
set_seed()
_sample_200 = random.sample(range(len(train_files)), min(200, len(train_files)))

pixel_data = []
for i in tqdm(_sample_200, desc="Loading pixels"):
    try:
        img = np.array(Image.open(train_files[i]).convert("RGB").resize(IMG_SIZE)) / 255.0
        pixel_data.append(img.reshape(-1, 3))
    except Exception:
        pass

pixel_data = np.concatenate(pixel_data, axis=0)  # (N*H*W, 3)
CHANNEL_MEAN = pixel_data.mean(axis=0)
CHANNEL_STD  = pixel_data.std(axis=0)

print(f"Dataset mean (R,G,B): {CHANNEL_MEAN.round(4)}")
print(f"Dataset std  (R,G,B): {CHANNEL_STD.round(4)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, col, ax) in enumerate(zip(["Red", "Green", "Blue"], ["red", "green", "blue"], axes)):
    ax.hist(pixel_data[:, i], bins=50, color=col, alpha=0.7, density=True)
    ax.set_title(f"{name} Channel")
    ax.set_xlabel("Pixel Value (normalised)")
    ax.set_ylabel("Density")
fig.suptitle("Per-Channel Pixel Intensity Distributions (200 training images)")
plt.tight_layout()
plt.savefig("figures/channel_histograms.png", dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Class Similarity Heatmap

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def _class_colour_hist(cls_name: str, n: int = 20) -> np.ndarray:
    """Mean 64-bin colour histogram over the first n images of a class."""
    cls_files = [f for f, l in zip(train_files, train_lbls) if CLASS_NAMES[l] == cls_name][:n]
    hists = []
    for fp in cls_files:
        try:
            img = np.array(Image.open(fp).convert("RGB").resize((64, 64)))
            h = np.concatenate([
                np.histogram(img[:, :, c], bins=64, range=(0, 256))[0]
                for c in range(3)
            ]).astype(np.float32)
            hists.append(h / (h.sum() + 1e-8))
        except Exception:
            pass
    return np.mean(hists, axis=0) if hists else np.zeros(192, dtype=np.float32)

print("Computing class colour histograms (may take ~30s)...")
class_hists = np.stack([_class_colour_hist(c) for c in tqdm(CLASS_NAMES)])
sim_matrix = cosine_similarity(class_hists)

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(sim_matrix, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cmap="YlOrRd", ax=ax, vmin=0, vmax=1)
ax.set_title("Pairwise Class Colour-Histogram Cosine Similarity")
plt.xticks(fontsize=5, rotation=90)
plt.yticks(fontsize=5)
plt.tight_layout()
plt.savefig("figures/class_similarity_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

## Section 4 — Data Preprocessing & Augmentation
### 4.1 Normalisation Values

In [ ]:
# Use dataset-computed mean/std; fall back to ImageNet if std is near zero (unreliable)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

if CHANNEL_STD.min() < 0.05:
    print("Dataset std values seem unreliable — falling back to ImageNet normalisation.")
    NORM_MEAN = IMAGENET_MEAN
    NORM_STD  = IMAGENET_STD
else:
    NORM_MEAN = CHANNEL_MEAN.tolist()
    NORM_STD  = CHANNEL_STD.tolist()
    print("Using dataset-computed normalisation.")

print(f"NORM_MEAN = {[f'{v:.4f}' for v in NORM_MEAN]}")
print(f"NORM_STD  = {[f'{v:.4f}' for v in NORM_STD]}")

### 4.2 – 4.3 Augmentation Pipelines

In [ ]:
train_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

eval_transform = T.Compose([
    T.Resize(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

print("Transforms defined: train (with augmentation), eval (normalise only).")

### 4.4 Custom Dataset & DataLoaders

In [ ]:
class FoodDataset(Dataset):
    def __init__(self, file_list: list, label_list: list, transform=None):
        self.files     = file_list
        self.labels    = label_list
        self.transform = transform

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, idx: int):
        try:
            img = Image.open(self.files[idx]).convert("RGB")
        except Exception as e:
            print(f"Warning: could not load image {self.files[idx]}: {e}")
            img = Image.new("RGB", (224, 224), color=(128, 128, 128))
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

In [ ]:
train_ds = FoodDataset(train_files, train_lbls, transform=train_transform)
val_ds   = FoodDataset(val_files,   val_lbls,   transform=eval_transform)
test_ds  = FoodDataset(test_files,  test_lbls,  transform=eval_transform)

_loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

train_loader = DataLoader(train_ds, shuffle=True,  **_loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_loader_kwargs)

print(f"DataLoaders ready — train: {len(train_loader)} batches, "
      f"val: {len(val_loader)} batches, test: {len(test_loader)} batches")

## Section 5 — Model Architecture (Transfer Learning)
### 5.1 Phase 1 — Feature Extraction (Frozen Backbone)

In [ ]:
def build_model(num_classes: int, device: torch.device) -> nn.Module:
    """EfficientNetB3 backbone (ImageNet pretrained) + custom classification head."""
    backbone = timm.create_model("efficientnet_b3", pretrained=True, num_classes=0)

    # Freeze all backbone parameters for Phase 1
    for param in backbone.parameters():
        param.requires_grad = False

    in_features = backbone.num_features  # 1536 for EfficientNetB3

    head = nn.Sequential(
        nn.BatchNorm1d(in_features),
        nn.Dropout(0.4),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes),
    )

    class FoodClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = backbone
            self.head     = head

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            features = self.backbone(x)  # timm returns global-pooled features when num_classes=0
            return self.head(features)

    model = FoodClassifier().to(device)

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params     : {total_params:,}")
    print(f"Trainable params : {trainable_params:,}")
    print(f"Frozen params    : {total_params - trainable_params:,}")
    return model

model = build_model(NUM_CLASSES, DEVICE)

### 5.2 Phase 2 — Fine-Tuning (Partial Backbone Unfreeze)
The `unfreeze_top_fraction()` helper is called after Phase 1 training converges.

In [ ]:
def unfreeze_top_fraction(model: nn.Module, fraction: float = 0.3) -> None:
    """Unfreeze the top `fraction` of backbone parameters (by count)."""
    backbone_params = list(model.backbone.named_parameters())
    n_to_unfreeze  = int(len(backbone_params) * fraction)
    for _, param in backbone_params[-n_to_unfreeze:]:
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"After unfreeze: {trainable:,} / {total:,} params trainable "
          f"({trainable / total * 100:.1f}%)")

## Section 6 — Training
### 6.1 Loss, Optimiser & Metrics

In [ ]:
import csv

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR
)

def topk_accuracy(outputs: torch.Tensor, targets: torch.Tensor, k: int) -> float:
    """Top-k accuracy for a batch."""
    _, topk_preds = outputs.topk(k, dim=1)
    correct = topk_preds.eq(targets.view(-1, 1).expand_as(topk_preds))
    return correct.any(dim=1).float().mean().item()

### 6.2 Callbacks: EarlyStopping, ReduceLROnPlateau, CSV Logger, Checkpoint

In [ ]:
class EarlyStopping:
    def __init__(self, patience: int = 5, mode: str = "min"):
        self.patience      = patience
        self.mode          = mode
        self.best          = float('inf') if mode == "min" else float('-inf')
        self.counter       = 0
        self.stop          = False
        self.best_weights  = None

    def step(self, metric: float, model: nn.Module) -> None:
        improved = metric < self.best if self.mode == "min" else metric > self.best
        if improved:
            self.best         = metric
            self.counter      = 0
            self.best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def restore(self, model: nn.Module) -> None:
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)


class CSVLogger:
    HEADER = ["epoch", "phase", "train_loss", "train_acc", "val_loss", "val_acc", "lr"]

    def __init__(self, path: str):
        self.path = path
        with open(path, 'w', newline='') as f:
            csv.writer(f).writerow(self.HEADER)

    def log(self, epoch: int, phase: str, train_loss: float, train_acc: float,
            val_loss: float, val_acc: float, lr: float) -> None:
        with open(self.path, 'a', newline='') as f:
            csv.writer(f).writerow([
                epoch, phase, f"{train_loss:.4f}", f"{train_acc:.4f}",
                f"{val_loss:.4f}", f"{val_acc:.4f}", f"{lr:.2e}"
            ])

### 6.3 Core Training Loop

In [ ]:
def run_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module,
              optimizer: optim.Optimizer, device: torch.device,
              is_train: bool) -> tuple[float, float, float]:
    """Run one epoch; return (loss, top1_acc, top3_acc)."""
    model.train() if is_train else model.eval()
    total_loss, top1_correct, top3_correct, total = 0.0, 0, 0, 0

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            bs           = images.size(0)
            total_loss  += loss.item() * bs
            top1_correct += (outputs.argmax(1) == labels).sum().item()
            top3_correct += round(topk_accuracy(outputs, labels, 3) * bs)
            total        += bs

    return total_loss / total, top1_correct / total, top3_correct / total

### 6.3 Phase 1 Training (Frozen Backbone)

In [ ]:
%%time
scheduler_p1 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=False
)
early_stop  = EarlyStopping(patience=5)
csv_logger  = CSVLogger("training_log.csv")

history = {
    "phase": [], "epoch": [], "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": [], "lr": []
}

print("=== Phase 1: Feature Extraction (frozen backbone) ===")
PHASE1_EPOCHS = 0
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, _ = run_epoch(model, train_loader, criterion, optimizer, DEVICE, True)
    vl_loss, vl_acc, _ = run_epoch(model, val_loader,   criterion, optimizer, DEVICE, False)
    current_lr = optimizer.param_groups[0]['lr']

    scheduler_p1.step(vl_loss)
    early_stop.step(vl_loss, model)

    csv_logger.log(epoch, "phase1", tr_loss, tr_acc, vl_loss, vl_acc, current_lr)
    history["phase"].append("phase1")
    history["epoch"].append(epoch)
    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)
    history["lr"].append(current_lr)

    print(f"Epoch {epoch:3d} | train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
          f"val_loss={vl_loss:.4f} val_acc={vl_acc:.4f} | lr={current_lr:.2e}")

    if early_stop.stop:
        print(f"Early stopping triggered at epoch {epoch}")
        break
    PHASE1_EPOCHS = epoch

early_stop.restore(model)
print(f"\nPhase 1 complete. Best val_loss: {early_stop.best:.4f}")

### 6.4 Phase 2 Fine-Tuning (Partial Backbone Unfreeze)

In [ ]:
%%time
unfreeze_top_fraction(model, fraction=0.3)
optimizer_p2 = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=FINE_TUNE_LR
)
scheduler_p2  = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_p2, mode='min', factor=0.5, patience=3, verbose=False
)
early_stop_p2 = EarlyStopping(patience=5)

FINE_TUNE_EPOCHS = 10
print(f"=== Phase 2: Fine-Tuning (top 30% backbone unfrozen) — up to {FINE_TUNE_EPOCHS} epochs ===")

for epoch in range(1, FINE_TUNE_EPOCHS + 1):
    global_epoch = PHASE1_EPOCHS + epoch
    tr_loss, tr_acc, _ = run_epoch(model, train_loader, criterion, optimizer_p2, DEVICE, True)
    vl_loss, vl_acc, _ = run_epoch(model, val_loader,   criterion, optimizer_p2, DEVICE, False)
    current_lr = optimizer_p2.param_groups[0]['lr']

    scheduler_p2.step(vl_loss)
    early_stop_p2.step(vl_loss, model)

    csv_logger.log(global_epoch, "phase2", tr_loss, tr_acc, vl_loss, vl_acc, current_lr)
    history["phase"].append("phase2")
    history["epoch"].append(global_epoch)
    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)
    history["lr"].append(current_lr)

    print(f"Epoch {global_epoch:3d} | train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
          f"val_loss={vl_loss:.4f} val_acc={vl_acc:.4f} | lr={current_lr:.2e}")

    if early_stop_p2.stop:
        print(f"Early stopping triggered at epoch {global_epoch}")
        break

early_stop_p2.restore(model)

try:
    torch.save(model.state_dict(), "best_food_classifier.pth")
    print("Model saved to best_food_classifier.pth")
except Exception as e:
    print(f"Warning: could not save model — {e}")

## Section 7 — Training Visualisation

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
df_hist = pd.DataFrame(history)

phase2_start_epoch = (
    df_hist[df_hist['phase'] == 'phase2']['epoch'].min()
    if 'phase2' in df_hist['phase'].values else None
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
ax = axes[0]
ax.plot(df_hist['epoch'], df_hist['train_loss'], label='Train Loss', marker='o', markersize=3)
ax.plot(df_hist['epoch'], df_hist['val_loss'],   label='Val Loss',   marker='o', markersize=3)
if phase2_start_epoch is not None:
    ax.axvline(phase2_start_epoch, color='gray', linestyle='--', label='Fine-tune start')
ax.set_title("Loss vs Epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()

# Accuracy curve
ax = axes[1]
ax.plot(df_hist['epoch'], df_hist['train_acc'], label='Train Acc', marker='o', markersize=3)
ax.plot(df_hist['epoch'], df_hist['val_acc'],   label='Val Acc',   marker='o', markersize=3)
if phase2_start_epoch is not None:
    ax.axvline(phase2_start_epoch, color='gray', linestyle='--', label='Fine-tune start')
ax.set_title("Accuracy vs Epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.legend()

# LR schedule
ax = axes[2]
ax.plot(df_hist['epoch'], df_hist['lr'], color='green', marker='o', markersize=3)
ax.set_title("Learning Rate Schedule")
ax.set_xlabel("Epoch")
ax.set_ylabel("LR")
ax.set_yscale('log')

fig.suptitle("Training Curves (both phases)", fontsize=14)
plt.tight_layout()
plt.savefig("figures/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

## Section 8 — Evaluation on Test Set
### 8.1 Load Best Model & Run Inference

In [ ]:
%%time
try:
    state = torch.load("best_food_classifier.pth", map_location=DEVICE)
    model.load_state_dict(state)
    print("Loaded best_food_classifier.pth")
except Exception as e:
    print(f"Warning: could not load checkpoint — {e}. Using current model weights.")

model.eval()
all_preds, all_probs, all_targets = [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Inference"):
        images  = images.to(DEVICE)
        outputs = torch.softmax(model(images), dim=1)
        preds   = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(outputs.cpu().numpy())
        all_targets.extend(labels.numpy())

all_preds   = np.array(all_preds)
all_probs   = np.array(all_probs)
all_targets = np.array(all_targets)
print(f"Inference complete. Test samples: {len(all_targets)}")

### 8.1 Core Metrics

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def topk_acc_np(probs: np.ndarray, targets: np.ndarray, k: int) -> float:
    topk = np.argsort(probs, axis=1)[:, -k:]
    return float(np.mean([t in row for t, row in zip(targets, topk)]))

top1 = float((all_preds == all_targets).mean())
top3 = topk_acc_np(all_probs, all_targets, 3)
top5 = topk_acc_np(all_probs, all_targets, 5)

prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
    all_targets, all_preds, average='macro', zero_division=0)
_, _, f1_weighted, _ = precision_recall_fscore_support(
    all_targets, all_preds, average='weighted', zero_division=0)
kappa = cohen_kappa_score(all_targets, all_preds)

metrics_df = pd.DataFrame({
    "Metric": ["Top-1 Acc", "Top-3 Acc", "Top-5 Acc",
               "Macro Precision", "Macro Recall", "Macro F1",
               "Weighted F1", "Cohen's Kappa"],
    "Value":  [f"{top1:.4f}", f"{top3:.4f}", f"{top5:.4f}",
               f"{prec_macro:.4f}", f"{rec_macro:.4f}", f"{f1_macro:.4f}",
               f"{f1_weighted:.4f}", f"{kappa:.4f}"]
})
print(metrics_df.to_string(index=False))

### 8.2 Per-Class Report

In [ ]:
print(classification_report(all_targets, all_preds, target_names=CLASS_NAMES, zero_division=0))

report_dict = classification_report(
    all_targets, all_preds, target_names=CLASS_NAMES,
    output_dict=True, zero_division=0
)
f1_per_class = {cls: report_dict[cls]['f1-score'] for cls in CLASS_NAMES}
sorted_f1    = sorted(f1_per_class.items(), key=lambda x: x[1], reverse=True)

print("\nTop-5 classes by F1:")
for cls, score in sorted_f1[:5]:
    print(f"  {cls:<30} F1={score:.4f}")

print("\nWorst-5 classes by F1:")
for cls, score in sorted_f1[-5:]:
    print(f"  {cls:<30} F1={score:.4f}")

### 8.3 Confusion Matrix

In [ ]:
cm = confusion_matrix(all_targets, all_preds)

fig, ax = plt.subplots(figsize=(24, 20))
sns.heatmap(cm, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cmap='Blues', ax=ax, annot=False)
ax.set_title("Confusion Matrix (61×61)", fontsize=14)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.xticks(fontsize=4, rotation=90)
plt.yticks(fontsize=4)
plt.tight_layout()
plt.savefig('figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
cm_offdiag = cm.copy()
np.fill_diagonal(cm_offdiag, 0)

top15_flat = np.argsort(cm_offdiag.flatten())[-15:][::-1]
pairs = []
for flat_idx in top15_flat:
    i, j = divmod(int(flat_idx), NUM_CLASSES)
    pairs.append((CLASS_NAMES[i], CLASS_NAMES[j], int(cm[i, j])))

pair_labels = [f"{p[0]}\n→{p[1]}" for p in pairs]
pair_counts = [p[2] for p in pairs]

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(range(len(pairs)), pair_counts, color='tomato')
ax.set_xticks(range(len(pairs)))
ax.set_xticklabels(pair_labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel("Count")
ax.set_title("15 Most-Confused Class Pairs")
plt.tight_layout()
plt.savefig('figures/confusion_matrix_zoomed.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.4 ROC / AUC (Macro)

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc as sklearn_auc

y_bin = label_binarize(all_targets, classes=list(range(NUM_CLASSES)))
per_class_aucs = []
for i in range(NUM_CLASSES):
    try:
        fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
        per_class_aucs.append(sklearn_auc(fpr, tpr))
    except Exception:
        per_class_aucs.append(0.5)

macro_auc = float(np.mean(per_class_aucs))
print(f"Macro-averaged AUC: {macro_auc:.4f}")

sorted_auc   = sorted(enumerate(per_class_aucs), key=lambda x: x[1], reverse=True)
top10_idx    = [i for i, _ in sorted_auc[:10]]
bottom10_idx = [i for i, _ in sorted_auc[-10:]]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, indices, title in zip(axes, [top10_idx, bottom10_idx], ["Top-10 AUC", "Bottom-10 AUC"]):
    for i in indices:
        fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
        ax.plot(fpr, tpr, label=f"{CLASS_NAMES[i]} ({per_class_aucs[i]:.2f})", linewidth=1)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.legend(fontsize=6, loc='lower right')
fig.suptitle(f"ROC Curves | Macro AUC = {macro_auc:.4f}", fontsize=13)
plt.tight_layout()
plt.savefig('figures/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.5 Misclassification Gallery

In [ ]:
set_seed()
wrong_indices = [i for i, (t, p) in enumerate(zip(all_targets, all_preds)) if t != p]
gallery_idx   = random.sample(wrong_indices, min(16, len(wrong_indices)))

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for ax, idx in zip(axes.flat, gallery_idx):
    try:
        img  = Image.open(test_files[idx]).convert("RGB").resize(IMG_SIZE)
        conf = all_probs[idx][all_preds[idx]] * 100
        ax.imshow(img)
        ax.set_title(
            f"True:  {CLASS_NAMES[all_targets[idx]]}\n"
            f"Pred:  {CLASS_NAMES[all_preds[idx]]} ({conf:.1f}%)",
            color='red', fontsize=7
        )
    except Exception:
        ax.set_title("(load error)", color='red', fontsize=7)
    ax.axis('off')
fig.suptitle("Misclassified Test Images (16 examples)", fontsize=13)
fig.tight_layout()
plt.savefig('figures/misclassification_gallery.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.6 Grad-CAM Visualisation

**Grad-CAM** (Gradient-weighted Class Activation Mapping) highlights the image regions that most influenced the model's prediction. It works by computing the gradient of the predicted class score with respect to the last convolutional feature map, then weighting those feature maps accordingly. Warmer colours (red/yellow) indicate regions the model attended to; cooler colours (blue) indicate regions it largely ignored.

In [ ]:
class GradCAM:
    """Minimal Grad-CAM targeting the last conv layer of the EfficientNetB3 backbone."""

    def __init__(self, model: nn.Module):
        self.model      = model
        self.gradients  = None
        self.activations= None
        target_layer    = model.backbone.conv_head  # last conv in EfficientNetB3

        target_layer.register_forward_hook(
            lambda m, inp, out: setattr(self, 'activations', out.detach())
        )
        target_layer.register_full_backward_hook(
            lambda m, grad_in, grad_out: setattr(self, 'gradients', grad_out[0].detach())
        )

    def __call__(self, image_tensor: torch.Tensor, class_idx: int) -> np.ndarray:
        self.model.eval()
        inp = image_tensor.unsqueeze(0).to(DEVICE).requires_grad_(True)

        output = self.model(inp)
        self.model.zero_grad()
        output[0, class_idx].backward()

        grads   = self.gradients.cpu().numpy()[0]    # (C, H, W)
        acts    = self.activations.cpu().numpy()[0]  # (C, H, W)
        weights = grads.mean(axis=(1, 2))            # (C,)
        cam     = np.maximum((weights[:, None, None] * acts).sum(axis=0), 0)
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam


gradcam = GradCAM(model)

# 4 correct + 4 incorrect predictions
correct_idx   = [i for i, (t, p) in enumerate(zip(all_targets, all_preds)) if t == p][:4]
incorrect_idx = gallery_idx[:4]
gcam_indices  = correct_idx + incorrect_idx

fig, axes = plt.subplots(2, 8, figsize=(24, 6))
for col, idx in enumerate(gcam_indices):
    img_tensor, label = test_ds[idx]
    pred_idx = int(all_preds[idx])
    try:
        cam = gradcam(img_tensor, pred_idx)
        orig_img  = np.array(Image.open(test_files[idx]).convert("RGB").resize(IMG_SIZE))
        cam_up    = np.array(Image.fromarray(
            (plt.cm.jet(cam)[:, :, :3] * 255).astype(np.uint8)
        ).resize(IMG_SIZE))
        overlay   = np.clip(0.5 * orig_img + 0.5 * cam_up, 0, 255).astype(np.uint8)
    except Exception as e:
        orig_img = np.zeros((*IMG_SIZE, 3), dtype=np.uint8)
        overlay  = orig_img.copy()

    is_correct = bool(all_targets[idx] == all_preds[idx])
    title_col  = "green" if is_correct else "red"
    label_str  = f"{'✓' if is_correct else '✗'} {CLASS_NAMES[pred_idx]}"

    axes[0, col].imshow(orig_img)
    axes[0, col].set_title(label_str, color=title_col, fontsize=7)
    axes[0, col].axis('off')

    axes[1, col].imshow(overlay)
    axes[1, col].set_title("Grad-CAM", fontsize=7)
    axes[1, col].axis('off')

fig.suptitle("Grad-CAM: Original (top row) vs Heatmap Overlay (bottom row)", fontsize=12)
plt.tight_layout()
plt.savefig('figures/gradcam_examples.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 9 — Inference Helper Function

In [ ]:
def predict_food(image_path: str, model: nn.Module, class_names: list,
                 top_k: int = 5) -> list[tuple[str, float]]:
    """
    Given a path to any food image, returns the top-k predicted classes
    with their probabilities.

    Args:
        image_path: Path to the image file.
        model: Trained classification model.
        class_names: List of class name strings.
        top_k: Number of top predictions to return.

    Returns:
        List of (class_name, probability) tuples sorted descending by probability.
    """
    try:
        img = Image.open(image_path).convert("RGB")
    except Exception as e:
        raise ValueError(f"Cannot open image at '{image_path}': {e}")

    tensor = eval_transform(img).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1).squeeze().cpu().numpy()

    top_indices = np.argsort(probs)[-top_k:][::-1]
    return [(class_names[i], float(probs[i])) for i in top_indices]

In [ ]:
set_seed()
demo_indices = random.sample(range(len(test_files)), 3)

for rank, idx in enumerate(demo_indices, 1):
    results    = predict_food(test_files[idx], model, CLASS_NAMES, top_k=5)
    true_label = CLASS_NAMES[test_lbls[idx]]
    print(f"
--- Demo {rank}: {pathlib.Path(test_files[idx]).name} ---")
    print(f"    True label: {true_label}")
    print("    Top-5 predictions:")
    for i, (cls, prob) in enumerate(results, 1):
        marker = "✓" if cls == true_label else " "
        print(f"      {i}. {marker} {cls:<35} {prob * 100:6.2f}%")

## Section 10 — Summary & Conclusions

### Approach

This notebook trained an **EfficientNetB3** backbone (pretrained on ImageNet) to classify food images into 61 categories using two-phase transfer learning:

1. **Phase 1 — Feature Extraction:** All backbone layers frozen; only the custom classification head (BatchNorm → Dropout(0.4) → Linear(1536→512) → ReLU → Dropout(0.3) → Linear(512→61)) is trained for up to 20 epochs with early stopping (patience=5).
2. **Phase 2 — Fine-Tuning:** The top 30% of backbone layers unfrozen; training continued with a 10× lower learning rate (1e-5) for up to 10 additional epochs.

Both phases used **Adam** optimiser with **ReduceLROnPlateau** (factor=0.5, patience=3) and **CrossEntropyLoss**.

---

### Final Metrics

| Metric | Value |
|--------|-------|
| Top-1 Accuracy | *(see Section 8.1 output)* |
| Top-3 Accuracy | *(see Section 8.1 output)* |
| Top-5 Accuracy | *(see Section 8.1 output)* |
| Macro F1-Score | *(see Section 8.1 output)* |
| Macro AUC | *(see Section 8.4 output)* |
| Cohen's Kappa | *(see Section 8.1 output)* |

---

### Key EDA Findings

- **Class imbalance:** Training image counts varied significantly across the 61 classes (see Section 3.1). An imbalance ratio > 3× was detected, suggesting class-weighted loss or oversampling may improve per-class recall on under-represented categories.
- **Image dimension variability:** Most images were near-square (see Section 3.3), but a notable tail of portrait and landscape images required resizing to 224×224 — this resize introduces mild distortion for extreme aspect ratios.
- **Visually similar classes:** The colour-histogram similarity heatmap (Section 3.5) revealed clusters of visually similar food groups (e.g., different rice dishes, different sushi types). These pairs dominate the off-diagonal of the confusion matrix and are the hardest to distinguish.

---

### Suggestions for Further Improvement

1. **Test-Time Augmentation (TTA):** Average softmax predictions across multiple augmented views (horizontal flips, slight crops, colour jitter) to reduce prediction variance at inference time.
2. **Mixup / CutMix Data Augmentation:** Linearly blend training image pairs and their labels. This regularises the model and improves generalisation on under-represented or visually similar classes.
3. **Label Smoothing:** Replace hard one-hot targets with soft labels (ε = 0.1) to reduce overconfidence and improve model calibration, especially useful when confused class pairs are semantically related.
4. **Larger Backbone:** EfficientNetB4/B5 or EfficientNetV2-M would increase model capacity at a modest compute cost and typically improves top-1 accuracy by 1–3 % over EfficientNetB3.
5. **Class-Weighted Loss:** Compute `weight = 1 / class_frequency` and pass to `nn.CrossEntropyLoss(weight=class_weights)` to directly penalise errors on rare classes more heavily.